In [11]:
using DifferentialEquations
using Plots
include("Chemkin.jl")


species_names = ["N2", "O2", "CH4", "H2O", "CO2"]
n_s = 5

#temperature (assumed constant)
T = 298.15 #K
P = 1e5 #Pa

R = 8.314 # J/molK

c_tot = P / (R*T) # ideal gas law

X0 = zeros(length(species_names)+1)
X0[1] = T # K
X0[2:end] = c_tot*[7.57/10.57, 2/10.57, 1/10.57, 0, 0] #mol/m^3, adiabatic mixture

# Stoichiometry: 2 O2 + CH4 -> 2 H2O + CO2
S = [0, -2, -1, 2, 1]  # N2 does not participate in the reaction
n_r = 1

species_dict = load_chemical_data("CHEMKIN-THERMDAT.txt")

Dict{String, Tuple{Float64, Vector{Float64}}} with 550 entries:
  "HSIC"          => (1500.0, [5.84954, 0.000762835, -9.97413e-8, -3.81159e-11,…
  "H2S"           => (1000.0, [2.88315, 0.00382783, -1.4234e-6, 2.498e-10, -1.6…
  "SIF3NHSIH3"    => (1000.0, [16.6994, 0.00778978, -8.11057e-7, -7.6502e-10, 1…
  "CLSI(CH3)2CH2" => (1500.0, [21.151, 0.00801827, -7.92425e-7, -3.29505e-10, 5…
  "GEF2"          => (1000.0, [4.76795, 0.00841094, -1.67642e-5, 1.56225e-8, -5…
  "CSICL3"        => (1500.0, [12.5054, 0.000533922, -2.58861e-7, 6.07531e-11, …
  "H2GAME"        => (600.0, [5.8316, 0.0122287, 3.03367e-7, -3.95694e-9, 1.225…
  "O2-"           => (1000.0, [3.88301, 0.000740787, -2.96178e-7, 5.7243e-11, -…
  "ASALME"        => (600.0, [7.12711, 0.00735786, 2.3008e-8, -2.2264e-9, 6.927…
  "CH2CLCHCL2"    => (1500.0, [16.1874, 0.00304768, -5.0115e-7, -1.5967e-11, 7.…
  "CHCLCCLOH"     => (1500.0, [14.1221, 0.00258376, -4.5769e-7, 5.21568e-12, 3.…
  "H2SI(CH3)CH2"  => (1500.0, [13.8883, 0.007

In [12]:
function Arrhenius(Y)
    T = Y[1]
    X = Y[2:end] * 1e-6 # Arrhenius takes /cm3
    # Constants from Westbrook & Dryer 1981
    n_CH4 = -0.3
    n_O2 = 1.3
    A = 1.3e8 
    Ea = 48.4e3 # cal/mol
    R = 1.987 # cal/mol/K
    
    # Ensure positive concentrations for exponentiation
    CH4_concentration = max(X[3], 0)
    O2_concentration = max(X[2], 0)
    
    k = A * exp(-Ea / (R * T)) # rate constant
    r = k * CH4_concentration^n_CH4 * O2_concentration^n_O2 # reaction rate
    return r * 1e6 # Back to /m3
end

Arrhenius (generic function with 1 method)

In [13]:
# Temperature rate of change due to reaction enthalpy
function dT(X, r)
    h0_vec = []
    cp_vec = []
    dH = sum( S .* [h0(X[1],species_dict[species_names[i]]) for i in 1:n_s] .* r)
    c_p = sum(X[2:end] .* [species_cp(X[1],species_dict[species_names[i]]) for i in 1:n_s])
    dT = -dH/c_p #J/mols * molK/J
    return dT[1]
end


dT (generic function with 1 method)

In [14]:
# Derivative function for the ODE system
function f!(dX, X, p, t)
    X = max.(1e-32,X)
    r = Arrhenius(X)
    dX[1] = dT(X,r)
    dX[2:end] .= r .* S  # Species concentrations change
end

# define timespan
tend = 1e32
tspan = (0, tend)

# define problem 
problem = ODEProblem(f!, X0, tspan)

# solve problem 
@time sol = solve(problem, AutoTsit5(Rosenbrock23()), abstol = 1e-10, reltol = 1e-8, maxiters=1e10)

  0.550168 seconds (1.10 M allocations: 51.603 MiB, 10.22% gc time, 98.01% compilation time: 61% of which was recompilation)


┌ Warning: At t=1.3883891487541548e24, dt was forced below floating point epsilon 2.68435456e8, and step error estimate = 1.279664933577007. Aborting. There is either an error in your model specification or the true solution is unstable (or the true solution can not be represented in the precision of Float64).
└ @ SciMLBase C:\Users\jelte\.julia\packages\SciMLBase\wfZCo\src\integrator_interface.jl:623


retcode: Unstable
Interpolation: specialized 4th order "free" interpolation, specialized 2nd order "free" stiffness-aware interpolation
t: 311-element Vector{Float64}:
      0.0
      9.999999999999999e-5
      0.0010999999999999998
      0.011099999999999997
      0.11109999999999996
      1.1110999999999995
     11.111099999999993
    111.11109999999994
   1111.1110999999994
  11111.111099999995
 111111.11109999994
      1.1111111110999994e6
      2.111111111109999e7
      ⋮
      1.3883891487541236e24
      1.3883891487541247e24
      1.3883891487541285e24
      1.3883891487541314e24
      1.3883891487541352e24
      1.3883891487541387e24
      1.3883891487541424e24
      1.388389148754146e24
      1.388389148754149e24
      1.388389148754152e24
      1.3883891487541545e24
      1.3883891487541548e24
u: 311-element Vector{Vector{Float64}}:
 [298.15, 28.89189702763006, 7.633262094486145, 3.8166310472430727, 0.0, 0.0]
 [298.15, 28.89189702763006, 7.633262094486145, 3.8166310472430727,

In [26]:
# define timespan
tend2 = 1e10
tspan = (0, tend2)
x02=sol[end]


# define problem 
problem = ODEProblem(f!, x02, tspan)

# solve problem 
@time sol2 = solve(problem, KenCarp47(), abstol = 1e-10, reltol = 1e-8, maxiters=1e10)

  0.168268 seconds (2.02 M allocations: 68.591 MiB, 28.29% gc time)


retcode: Success
Interpolation: 3rd order Hermite
t: 1747-element Vector{Float64}:
      0.0
      2.302467737722909
     25.327145114952
    255.57391888724288
   2558.0416566101517
  25582.719033839236
 255829.49280613006
      2.5582972305290382e6
      1.1437493069039607e7
      1.5978799999130834e7
      2.8228316762213044e7
      4.047783352529526e7
      5.67459214043861e7
      ⋮
      9.109157936350422e9
      9.109157937080109e9
      9.109157944376974e9
      9.109158017345623e9
      9.109158747032108e9
      9.109166043896957e9
      9.10923901254544e9
      9.10996869903028e9
      9.117265563878664e9
      9.190234212362513e9
      9.919920697201e9
      1.0e10
u: 1747-element Vector{Vector{Float64}}:
 [509.72608592356954, 28.89189702763006, 6.979728620876648, 3.489864310438324, 0.6535334736094899, 0.32676673680474494]
 [509.726085926412, 28.89189702763006, 6.9797286208675775, 3.4898643104337888, 0.6535334736185605, 0.32676673680928026]
 [509.72608595483644, 28.891897027

In [28]:
t = sol.t
T = sol[1, :]
t2 = sol2.t
T2 = sol2[1, :]

# Create plots with half height
plot1 = plot(t,T, xlabel="Time (s)", ylabel="T (K)", legend=false)
plot2 = plot(t2,T2, xlabel="Time (s)", ylabel="T (K)", legend=false)

# Combine plots
combined_plot = plot(plot1, plot2, layout=(2, 1))

savefig("1stepthermo2")

"C:\\Users\\jelte\\chemicalcombustion\\Toy Models\\1stepthermo2.png"

In [7]:
using LinearAlgebra: norm

function compute_relaxation_time(sol; tol=1e-6)
    """
    Compute relaxation time τ for a solution `sol` of an ODEProblem.
    
    τ is defined as the time when ‖u(t) - u_steady‖ / ‖u0 - u_steady‖ ≈ 1/e.
    
    Args:
        sol: Solution object from DifferentialEquations.jl
        tol: Tolerance for checking convergence (default: 1e-6)
    
    Returns:
        τ: Relaxation time (first time when decay reaches 1/e)
        If no such time is found, returns `nothing`.
    """
    u0 = sol.prob.u0          # Initial (perturbed) state
    u_steady = sol[end]       # Equilibrium state (final value)
    Δ0 = norm(u0 - u_steady)  # Initial deviation magnitude
    
    # Handle cases where Δ0 ≈ 0 (no perturbation)
    if Δ0 < tol
        @warn "Initial state is already at equilibrium (‖Δu‖ = $Δ0). τ is undefined."
        return nothing
    end
    
    # Target deviation: Δ0 / e
    target_deviation = Δ0 / MathConstants.e
    
    # Find the first time when ‖u(t) - u_steady‖ ≤ target_deviation
    τ = nothing
    for (i, t) in enumerate(sol.t)
        Δu = norm(sol.u[i] - u_steady)
        if Δu ≤ target_deviation + tol  # Allow numerical tolerance
            τ = t
            break
        end
    end
    
    if τ === nothing
        @warn "No relaxation time found within solution timeframe. Increase `tspan`."
    end
    
    return τ
end

compute_relaxation_time(sol)

-6.548300274163072e19